# SS3 Part B: STADIO 1 subscriber churn prediction on the client data extract

**Method applied:** the final method chosen in SS2: a gradient-boosted tree classifier (XGBoost), compared against the logistic-regression baseline and the simple "no recent viewing" rule from the SS1 problem statement.
**Task:** predict which STADIO 1 TV subscribers will cancel (voluntarily, by payment failure or by non-renewal) in the next 30 days, early enough for a retention action to change the outcome.

The notebook has five components, each clearly marked with a heading and committed separately:

| # | Component | Marks |
|---|---|---|
| 1 | Dataset loading | 2 |
| 2 | Dataset preprocessing (incl. train / validation / test split) | 5 |
| 3 | Model training (incl. hyper-parameter justification) | 5 |
| 4 | Model validation (unseen test subset) | 5 |
| 5 | Results visualisation (training and unseen) | 3 |

Run top to bottom. Set `STADIO_DATA_DIR` (or place the CSVs in `data/raw/stadioone/`) if your data lives elsewhere.

---
# 1. DATASET LOADING
*(Component 1 of 5)*

In [ ]:
import json, os, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 40)
SEED = 42
np.random.seed(SEED)

# Look for the client extract in a few sensible places (first match wins).
CANDIDATES = [os.environ.get("STADIO_DATA_DIR", ""), "data/raw/stadioone", "data/raw", "stadioone",
              "iitaemmanuel_client_data/stadioone", "../data/raw/stadioone"]
DATA_DIR = next((Path(p) for p in CANDIDATES if p and (Path(p) / "table1_subscriptions.csv").exists()), None)
assert DATA_DIR is not None, "Client CSVs not found: set STADIO_DATA_DIR or copy them to data/raw/stadioone/"
OUT_DIR = Path(os.environ.get("STADIO_OUT_DIR", "outputs")); (OUT_DIR / "figures").mkdir(parents=True, exist_ok=True)
print("Reading client data from:", DATA_DIR.resolve())

FILES = {"subs": "table1_subscriptions", "pay": "table2_billing_payments", "view": "table3_viewing_sessions",
         "cal": "table4_race_calendar", "app": "table5_app_fantasy_activity", "tix": "table6_ticketing_hospitality",
         "mail": "table7_email_marketing", "social": "table8_social_comments", "prof": "table9a_fan_profile",
         "merch": "table9b_merchandise", "idmap": "table10_identity_mapping"}
# Everything is read as text first: the extract is uncleaned (mixed date formats, casing, stray spaces),
# so typing is done explicitly in the preprocessing component.
raw = {k: pd.read_csv(DATA_DIR / f"{f}.csv", dtype=str, keep_default_na=True) for k, f in FILES.items()}
pd.DataFrame({"table": [FILES[k] for k in raw], "rows": [len(v) for v in raw.values()],
              "columns": [v.shape[1] for v in raw.values()]}).set_index("table")

In [ ]:
# Which tables does the project need? (SS2 data request: churn label from table 1; features from tables 2-7, 9a, 9b, 10; calendar from 4.)
# Table 8 (social posts) is only a 38%-linked *sample*, so absence of posts is not evidence of disengagement: it is left out.
subs0 = raw["subs"].copy()
print("Subscription periods (raw):", len(subs0), "| distinct subscribers:", subs0.subscriber_id.nunique())
print("\nMissing values per column in table 1:\n", subs0.isna().sum()[lambda s: s > 0])
print("\nExact duplicate rows in table 1:", subs0.duplicated().sum())

# Class balance at the subscription-period level, using the churn definition agreed for this project:
# churn = the fan asked to cancel (status Cancelled / Payment failed, i.e. a cancellation_datetime exists).
# Expired periods ended naturally; cancellation_type separates voluntary / payment failure / non-renewal.
s = subs0.drop_duplicates().copy()
s["status_c"] = s.status.str.strip().str.title()
s["churned_period"] = s.cancellation_datetime.notna().astype(int)
print("\nStatus x churn flag:\n", pd.crosstab(s.status_c, s.churned_period))
print("\nChurn proportion over subscription periods: %.1f%%" % (100 * s.churned_period.mean()))
raw["subs"].head()

### What was loaded (dataset summary)

**Source.** 11 CSV extracts from the client (all loaded as text and typed explicitly later). The unit of analysis is built from table 1 (`subscriptions`): **5,837 raw rows = 5,820 distinct subscription periods from 3,800 subscribers** (17 exact duplicates, removed in preprocessing). The other tables are behavioural context: viewing sessions (56,084), payments (6,322), app/fantasy events (43,130), tickets (1,418), emails (29,665), merchandise (2,584), fan profile (3,811), race calendar (61) and the identity map (3,811). Table 8 (social posts) is loaded but not modelled: it is only a sample, and only ~38% of posts can be linked to a subscriber, so "no posts" cannot be read as disengagement.

**Target and classes.** Churn = the fan asked to cancel, had a payment failure, or did not renew (status `Cancelled` / `Payment failed`, i.e. a `cancellation_datetime` exists). Expired periods ended naturally. At subscription-period level this gives **1,340 churned (23.0%) vs 4,497 not churned (77.0%)** (the 58 `Active` periods have not ended).

**Modelling instances.** The prediction task is "will this subscriber cancel in the next 30 days?", scored once per month for every running subscription. After preprocessing this gives **22,958 subscriber-month instances (3,277 subscribers)** with two classes: *stays* (0) and *cancels within 30 days* (1). The **imbalance is about 96.2 : 3.8 (roughly 25 : 1; 871 positives)**, so accuracy is meaningless (always predicting "stays" scores 96%) and the evaluation uses PR-AUC, recall/precision and top-decile lift instead.

**Data quality found in the extract** (the client warned it is uncleaned): mixed casing/whitespace in categories, four different datetime spellings, exact duplicate rows in most tables, negative plan prices, impossible session durations (e.g. -1 s or 7 days) and watch percentages outside 0-100, and 608 low-confidence (<0.9) identity links.

---
# 2. DATASET PREPROCESSING
*(Component 2 of 5)*

### Why this particular preprocessing?

| Decision | What was done | Why |
|---|---|---|
| **Typing and cleaning** | Four datetime formats parsed explicitly (unparseable values are counted); categories de-cased and de-spaced; duplicate rows/IDs dropped; negative prices set to the plan median; session durations outside 0-24 h and watch % outside 0-100 set to missing | The client states the extract is raw operational data. Un-normalised "Smart TV"/"smart tv" would be treated as different devices and bad durations would distort the engagement features. |
| **Unit of analysis: subscriber-month** | On the 1st of each month (Jan 2024 - Jul 2026), every subscriber with a running, not-yet-cancelled period is one row; label = cancels in the next **30 days** | A model is only useful if it scores people *before* they leave. This copies the SS1/SS2 design (a prediction date per subscriber, features before it, label after it) and handles monthly, seasonal and annual plans alike. Observation dates start in Jan 2024 so every row has >= 90 days of history; the last date leaves a full 30-day label window before the data ends. |
| **Horizon = 30 days** | Lower end of the 30-60 day window agreed in SS1 | In this data cancellations happen in the last 30 days of a period, and the behavioural change is visible in the final weeks. In a trial run, a 60-day horizon gave a clearly weaker test ROC-AUC (about 0.70 vs 0.75). |
| **Leakage control** | Every feature uses only events *strictly before* the observation date. `status`, `end_datetime`, `cancellation_*` and `auto_renew` (recorded at period end) are never used, and the contract term comes from `plan_type` instead of `end_datetime`. `notification_setting` (state at extraction) is dropped. An assertion in the notebook checks this. | The client README warns these fields reveal the outcome; including them would give a spectacular but worthless model. |
| **Retention-offer fans excluded** | Subscriber-months with a "Retention offer" email in the previous 90 days are removed (1,461 rows), and retention emails are not used as features | The business sends those offers *because* its own score flagged the fan, so their behaviour is not natural and the email would encode the outcome (client guidance). |
| **Identity links** | App and ticketing data are joined only where `match_confidence >= 0.9` | Low-confidence (probabilistic) joins would attach another person's behaviour to a subscriber (SS1 risk R1). |
| **Features** | Recency, frequency, volume and trend of viewing (7-90 day windows, session length, % watched, 30-vs-90-day trends), payment failures/refunds, plan and price state, tenure and previous churn, app and email engagement, ticket no-shows, merchandise, profile, and race-calendar context | These are the SS2 feature groups (relative to the prediction date, following Gregory, 2018) adapted to the STADIO 1 tables. |
| **Missing values** | Counts/sums = 0 when there are no events; "days since" = 181 plus a `has_viewing_180d` flag; other gaps left missing | No activity is genuine information. XGBoost handles NaN natively; for logistic regression the imputer adds missing-indicator columns. |
| **Time-based split, no shuffling** | Train Jan-Dec 2024, validation Jan-Jun 2025, **test Jul 2025 - Jul 2026 (unseen)**; an embargo guarantees no training label window overlaps the next subset | A random split would mix the future into training and overstate performance. A full held-out year also covers the seasonal calendar (SS1 risk R3). The same fan can appear in several months, which is realistic (fans are re-scored every month), but never in the same month across sets. |
| **No re-sampling or class weights** | Natural 4% churn rate kept | SS2 followed Ahmad et al. (2019): boosted trees do well without re-balancing and probabilities stay calibrated (checked by the calibration plot). |
| **Encoders fitted on training data only** | Tree model: ordinal-encode categoricals; logistic regression: median impute + missing flags, signed log1p, standardise, one-hot | Fitting on all data would leak test statistics into training. |

In [ ]:
# ---- 2a. Cleaning helpers (the extract is live-system data and has NOT been cleaned) --------------------------
DT_FORMATS = ["%Y-%m-%d %H:%M:%S", "%d-%b-%Y %H:%M:%S", "%d/%m/%Y %H:%M", "%Y/%m/%d %H:%M:%S", "%Y-%m-%d"]

def parse_dt(s):
    """Parse the datetime spellings found in the extract; anything unparseable becomes NaT (and is counted)."""
    s = s.astype("string").str.strip()
    out = pd.Series(pd.NaT, index=s.index, dtype="datetime64[ns]")
    for f in DT_FORMATS:
        m = out.isna() & s.notna()
        if m.any():
            out[m] = pd.to_datetime(s[m], format=f, errors="coerce")
    bad = int((out.isna() & s.notna()).sum())
    return out, bad

def canon(s):
    """Fix casing / stray whitespace: map every spelling to the most frequent spelling of the same value."""
    s = s.astype("string").str.strip()
    key = s.str.lower()
    best = s.groupby(key).agg(lambda x: x.value_counts().index[0])
    return key.map(best)

def to_bool(s):
    return s.astype("string").str.strip().str.upper().map({"TRUE": True, "FALSE": False}).astype("boolean")

report = {}   # data-quality log, printed at the end of the cell

# ---- table 1: subscriptions -------------------------------------------------------------------------------------
subs = raw["subs"].copy()
for c in ["start_datetime", "end_datetime", "cancellation_datetime"]:
    subs[c], bad = parse_dt(subs[c]); report[f"unparseable {c}"] = bad
for c in ["plan_type", "status", "cancellation_type", "acquisition_channel", "promo_code", "currency"]:
    subs[c] = canon(subs[c])
for c in ["plan_price", "discount_amount"]:
    subs[c] = pd.to_numeric(subs[c], errors="coerce")
n0 = len(subs); subs = subs.drop_duplicates().drop_duplicates("subscription_id"); report["duplicate subscription rows dropped"] = n0 - len(subs)
bad_dates = subs.end_datetime.notna() & (subs.end_datetime <= subs.start_datetime)
report["periods with end <= start dropped"] = int(bad_dates.sum()); subs = subs[~bad_dates]
neg = subs.plan_price < 0; report["negative plan_price set to missing"] = int(neg.sum()); subs.loc[neg, "plan_price"] = np.nan
subs["plan_price"] = subs.plan_price.fillna(subs.groupby("plan_type").plan_price.transform("median"))
subs["discount_amount"] = subs.discount_amount.fillna(0).clip(lower=0)

# Contractual term (days) is fixed by the plan type and known at sign-up; the *actual* end_datetime is only known at the outcome.
TERM_DAYS = {"Monthly": 30, "Trial": 30, "Seasonal": 180, "Annual": 360}
subs["term_days"] = subs.plan_type.map(TERM_DAYS)
# TARGET (SS1/SS2 definition): the fan cancelled, failed to pay or did not renew -> a cancellation_datetime exists.
subs["churned"] = subs.cancellation_datetime.notna().astype(int)

# ---- table 2: payments ----------------------------------------------------------------------------------------------
pay = raw["pay"].copy(); pay["payment_datetime"], bad = parse_dt(pay.payment_datetime); report["unparseable payment_datetime"] = bad
pay["payment_status"] = canon(pay.payment_status).str.title(); pay["payment_method_type"] = canon(pay.payment_method_type)
pay["amount"] = pd.to_numeric(pay.amount, errors="coerce")
n0 = len(pay); pay = pay.drop_duplicates().drop_duplicates("payment_id"); report["duplicate payments dropped"] = n0 - len(pay)
pay = pay.merge(subs[["subscription_id", "subscriber_id"]], on="subscription_id", how="inner")

# ---- table 3: viewing sessions -----------------------------------------------------------------------------------
view = raw["view"].copy(); view["session_start_datetime"], bad = parse_dt(view.session_start_datetime); report["unparseable session_start"] = bad
for c in ["content_type", "live_or_on_demand", "device_type"]: view[c] = canon(view[c])
view["session_duration"] = pd.to_numeric(view.session_duration, errors="coerce")
view["percent_watched"] = pd.to_numeric(view.percent_watched, errors="coerce")
n0 = len(view); view = view.drop_duplicates().drop_duplicates("session_id"); report["duplicate sessions dropped"] = n0 - len(view)
bad_dur = ~view.session_duration.between(0, 86_400); report["session_duration outside [0, 24h] set to missing"] = int(bad_dur.sum())
view.loc[bad_dur, "session_duration"] = np.nan
bad_pct = ~view.percent_watched.between(0, 100); report["percent_watched outside [0, 100] set to missing"] = int(bad_pct.sum())
view.loc[bad_pct, "percent_watched"] = np.nan
view = view.dropna(subset=["session_start_datetime"])

# ---- table 4: race calendar ----------------------------------------------------------------------------------
cal = raw["cal"].copy(); cal["event_date"], bad = parse_dt(cal.event_date); report["unparseable event_date"] = bad
cal = cal.drop_duplicates("event_id").sort_values("event_date")

# ---- table 5: app / fantasy ------------------------------------------------------------------------------------
app = raw["app"].copy(); app["activity_datetime"], bad = parse_dt(app.activity_datetime); report["unparseable activity_datetime"] = bad
app["activity_type"] = canon(app.activity_type)
n0 = len(app); app = app.drop_duplicates(); report["duplicate app events dropped"] = n0 - len(app)
app = app.dropna(subset=["activity_datetime"])
# NB: `notification_setting` is the state at the extraction date (after the outcome) so it is NOT used as a feature.

# ---- table 6: ticketing --------------------------------------------------------------------------------------------
tix = raw["tix"].drop_duplicates("ticket_id").copy(); tix["purchase_datetime"], bad = parse_dt(tix.purchase_datetime); report["unparseable purchase_datetime"] = bad
tix["attended"] = to_bool(tix.attended_flag); tix["price_paid"] = pd.to_numeric(tix.price_paid, errors="coerce")
tix = tix.merge(cal[["event_id", "event_date"]], on="event_id", how="left")

# ---- table 7: email -------------------------------------------------------------------------------------------------
mail = raw["mail"].copy(); mail["send_datetime"], bad = parse_dt(mail.send_datetime); report["unparseable send_datetime"] = bad
mail["campaign_type"] = canon(mail.campaign_type).str.title()
for c in ["opened", "clicked", "unsubscribed", "offer_taken"]: mail[c] = to_bool(mail[c]).fillna(False).astype(int)
n0 = len(mail); mail = mail.drop_duplicates().drop_duplicates("send_id"); report["duplicate emails dropped"] = n0 - len(mail)
mail = mail.dropna(subset=["send_datetime"])

# ---- tables 9a / 9b / 10 ---------------------------------------------------------------------------------------------
prof = raw["prof"].drop_duplicates("subscriber_id").copy()
prof["marketing_consent"] = to_bool(prof.marketing_consent).fillna(False).astype(int)
prof["has_favourite_team"] = prof.favourite_team.notna().astype(int)
merch = raw["merch"].copy(); merch["merch_purchase_datetime"], bad = parse_dt(merch.merch_purchase_datetime)
merch["merch_amount"] = pd.to_numeric(merch.merch_amount, errors="coerce"); merch = merch.drop_duplicates().dropna(subset=["merch_purchase_datetime"])

# Identity resolution: a join is trusted only if match_confidence >= 0.9; weaker (probabilistic) links are treated as "no link".
CONF_MIN = 0.9
idm = raw["idmap"].copy(); idm["match_confidence"] = pd.to_numeric(idm.match_confidence, errors="coerce")
idm = idm.sort_values("match_confidence", ascending=False).drop_duplicates("subscriber_id")
report["id links below confidence 0.9 ignored"] = int((idm.match_confidence < CONF_MIN).sum())
idm = idm[idm.match_confidence >= CONF_MIN]
app_map = idm.dropna(subset=["app_user_id"]).drop_duplicates("app_user_id")[["subscriber_id", "app_user_id"]]
tix_map = idm.dropna(subset=["ticketing_customer_id"]).drop_duplicates("ticketing_customer_id")[["subscriber_id", "ticketing_customer_id"]]
app = app.merge(app_map, on="app_user_id", how="inner")
tix = tix.merge(tix_map, left_on="customer_id", right_on="ticketing_customer_id", how="inner")

pd.Series(report, name="count").to_frame()

In [ ]:
# ---- 2b. Build the modelling table: one row per (subscriber, observation date) ----------------------------------
# An observation date is the 1st of a month. A subscriber is "at risk" on that date if a subscription period is running
# and has not yet been cancelled. Label = 1 if that period is cancelled within the next HORIZON days (churn in the window
# AFTER the observation date). All features below use only events strictly BEFORE the observation date (no leakage).
HORIZON = 30                      # days ahead to predict (lower end of the 30-60 day window from SS1; see markdown for why)
DATA_END = pd.Timestamp("2026-08-31")
OBS_DATES = pd.date_range("2024-01-01", DATA_END - pd.Timedelta(days=HORIZON), freq="MS")   # >=90 days of history, full label window

snap = []
for t in OBS_DATES:
    run = subs[(subs.start_datetime < t) & (subs.end_datetime > t) & ~(subs.cancellation_datetime <= t)].copy()
    run["obs_date"] = t
    snap.append(run)
snap = pd.concat(snap, ignore_index=True)
snap["y"] = (snap.cancellation_datetime.notna() & (snap.cancellation_datetime > snap.obs_date) &
             (snap.cancellation_datetime <= snap.obs_date + pd.Timedelta(days=HORIZON))).astype(int)
snap["row_id"] = np.arange(len(snap))
print(f"{len(snap):,} subscriber-months at risk from {len(OBS_DATES)} observation dates; positives: {snap.y.sum():,} ({snap.y.mean():.1%})")

# Retention-offer emails are triggered by the business's own churn score and change behaviour, so those fans are not "natural"
# (client guidance). We exclude subscriber-months where a retention offer was sent in the 90 days before the observation date.
ret = mail[mail.campaign_type == "Retention Offer"][["subscriber_id", "send_datetime"]]
m = snap[["row_id", "subscriber_id", "obs_date"]].merge(ret, on="subscriber_id")
m = m[(m.send_datetime < m.obs_date) & (m.send_datetime >= m.obs_date - pd.Timedelta(days=90))]
retention_rows = set(m.row_id)
print(f"Excluding {len(retention_rows):,} subscriber-months with a retention offer in the previous 90 days "
      f"(positive rate among them: {snap.loc[snap.row_id.isin(retention_rows), 'y'].mean():.1%})")
snap = snap[~snap.row_id.isin(retention_rows)].reset_index(drop=True)
snap["row_id"] = np.arange(len(snap))
print(f"Modelling table: {len(snap):,} rows, {snap.subscriber_id.nunique():,} subscribers, positive rate {snap.y.mean():.2%}")

In [ ]:
# ---- 2c. Feature engineering (every feature is computed relative to the row's observation date) --------------------
def events_before(ev, key, tcol, max_days=None):
    """Join events to the modelling rows and keep those strictly before obs_date (optionally within max_days)."""
    m = snap[["row_id", "subscriber_id", "obs_date"]].merge(ev, left_on="subscriber_id", right_on=key, how="inner")
    m["age"] = (m.obs_date - m[tcol]).dt.total_seconds() / 86400
    m = m[m.age > 0]
    return m if max_days is None else m[m.age <= max_days]

F = snap[["row_id"]].copy().set_index("row_id")
def add(df):
    global F; F = F.join(df, how="left")

# --- (i) subscription / plan state (nothing from status, end_datetime, cancellation_*, auto_renew: those leak the outcome)
d = snap.set_index("row_id")
sub_f = pd.DataFrame({
    "plan_type": d.plan_type, "acquisition_channel": d.acquisition_channel, "plan_price": d.plan_price,
    "discount_ratio": (d.discount_amount / d.plan_price).clip(0, 1), "has_promo": d.promo_code.notna().astype(int),
    "term_days": d.term_days,
    "days_since_start": (d.obs_date - d.start_datetime).dt.days,
})
sub_f["frac_term_elapsed"] = sub_f.days_since_start / sub_f.term_days
sub_f["days_to_planned_end"] = sub_f.term_days - sub_f.days_since_start
sub_f["obs_month"] = d.obs_date.dt.month
# history of earlier periods of the same fan that had already finished before this period started
hist = snap[["row_id", "subscriber_id", "subscription_id", "start_datetime"]].merge(
    subs[["subscriber_id", "subscription_id", "start_datetime", "end_datetime", "churned"]], on="subscriber_id", suffixes=("", "_p"))
hist = hist[hist.end_datetime.notna() & (hist.end_datetime <= hist.start_datetime)]
g = hist.groupby("row_id")
add(pd.DataFrame({"n_prior_periods": g.size(), "n_prior_churned": g.churned.sum(),
                  "tenure_days": (snap.set_index("row_id").start_datetime - g.start_datetime_p.min()).dt.days,
                  "days_since_prev_end": (snap.set_index("row_id").start_datetime - g.end_datetime.max()).dt.days}))
F = F.join(sub_f); F[["n_prior_periods", "n_prior_churned"]] = F[["n_prior_periods", "n_prior_churned"]].fillna(0)
F["tenure_days"] = F.tenure_days.fillna(F.days_since_start)

# --- (ii) billing: payment problems before the observation date
p = events_before(pay, "subscriber_id", "payment_datetime")
fail = p[p.payment_status == "Failed"]; bad = p[p.payment_status.isin(["Refunded", "Chargeback"])]
add(pd.DataFrame({"n_failed_payments": fail.groupby("row_id").size(),
                  "n_failed_90d": fail[fail.age <= 90].groupby("row_id").size(),
                  "n_refund_chargeback": bad.groupby("row_id").size(),
                  "days_since_last_failure": fail.groupby("row_id").age.min(),
                  "n_payments": p.groupby("row_id").size()}))
last_pay = p.sort_values("age").groupby("row_id").first()
add(last_pay[["payment_method_type"]].rename(columns={"payment_method_type": "last_payment_method"}))
for c in ["n_failed_payments", "n_failed_90d", "n_refund_chargeback", "n_payments"]: F[c] = F[c].fillna(0)

# --- (iii) viewing behaviour: recency, frequency, volume, depth and trend (the strongest signal in SS2)
v = events_before(view, "subscriber_id", "session_start_datetime", 180)
v["hours"] = v.session_duration / 3600; v["day"] = v.session_start_datetime.dt.normalize()
for w in [7, 14, 30, 60, 90]:
    x = v[v.age <= w]; gx = x.groupby("row_id")
    add(pd.DataFrame({f"sessions_{w}d": gx.size(), f"hours_{w}d": gx.hours.sum(), f"active_days_{w}d": gx.day.nunique()}))
x = v[v.age <= 90]; gx = x.groupby("row_id")
x30 = v[v.age <= 30]
add(pd.DataFrame({"avg_session_min_30d": x30.groupby("row_id").session_duration.mean() / 60,
                  "avg_session_min_90d": gx.session_duration.mean() / 60,
                  "avg_pct_watched_30d": x30.groupby("row_id").percent_watched.mean()}))
add(pd.DataFrame({"avg_pct_watched_90d": gx.percent_watched.mean(), "live_share_90d": gx.apply(lambda a: (a.live_or_on_demand == "Live").mean()),
                  "n_devices_90d": gx.device_type.nunique(),
                  "race_sessions_90d": x[x.content_type.isin(["Race", "Qualifying", "Sprint"])].groupby("row_id").size()}))
add(pd.DataFrame({"days_since_last_session": v.groupby("row_id").age.min()}))
for c in [c for c in F.columns if c.startswith(("sessions_", "hours_", "active_days_", "race_sessions_"))]: F[c] = F[c].fillna(0)
F["n_devices_90d"] = F.n_devices_90d.fillna(0)
F["has_viewing_180d"] = (F.days_since_last_session.notna()).astype(int)
F["days_since_last_session"] = F.days_since_last_session.fillna(181)
# trends: activity in the last 14 days vs the 14 days before that, and the last 30 days vs the 90-day average pace
prev14 = F.sessions_30d - F.sessions_14d
F["trend_sessions_14d"] = (F.sessions_14d - (F.sessions_30d - F.sessions_14d)) / (F.sessions_30d + 1)
F["trend_hours_30_vs_90"] = F.hours_30d / (F.hours_90d / 3 + 0.5)
F["trend_sessions_30_vs_90"] = F.sessions_30d / (F.sessions_90d / 3 + 0.5)
F["trend_session_len_30_vs_90"] = F.avg_session_min_30d / F.avg_session_min_90d

# --- (iv) app and fantasy activity (only fans with a trusted app link)
a = events_before(app, "subscriber_id", "activity_datetime", 90)
for w in [30, 90]:
    x = a[a.age <= w]; add(pd.DataFrame({f"app_events_{w}d": x.groupby("row_id").size()}))
add(pd.DataFrame({"app_logins_30d": a[(a.age <= 30) & (a.activity_type == "Login")].groupby("row_id").size(),
                  "app_types_90d": a.groupby("row_id").activity_type.nunique(),
                  "days_since_last_app": a.groupby("row_id").age.min()}))
F["has_app_link"] = snap.set_index("row_id").subscriber_id.isin(app_map.subscriber_id).astype(int)
for c in ["app_events_30d", "app_events_90d", "app_logins_30d", "app_types_90d"]: F[c] = F[c].fillna(0)

# --- (v) ticketing: purchases and no-shows (no-show is only known once the event date has passed)
tk = events_before(tix, "subscriber_id", "purchase_datetime")
past = tk[tk.event_date < tk.obs_date]
add(pd.DataFrame({"n_tickets": tk.groupby("row_id").size(), "ticket_spend": tk.groupby("row_id").price_paid.sum(),
                  "n_no_shows": past[past.attended == False].groupby("row_id").size()}))
for c in ["n_tickets", "ticket_spend", "n_no_shows"]: F[c] = F[c].fillna(0)
F["has_ticket_link"] = snap.set_index("row_id").subscriber_id.isin(tix_map.subscriber_id).astype(int)

# --- (vi) email responsiveness (retention offers excluded; only consented fans appear in this table)
e = events_before(mail[mail.campaign_type != "Retention Offer"], "subscriber_id", "send_datetime", 90)
ge = e.groupby("row_id")
add(pd.DataFrame({"emails_90d": ge.size(), "open_rate_90d": ge.opened.mean(), "click_rate_90d": ge.clicked.mean(),
                  "renewal_reminders_90d": e[e.campaign_type == "Renewal Reminder"].groupby("row_id").size()}))
unsub = events_before(mail[mail.unsubscribed == 1], "subscriber_id", "send_datetime").groupby("row_id").size()
F["unsubscribed_before"] = unsub.reindex(F.index).fillna(0).clip(upper=1)
F["in_email_table"] = snap.set_index("row_id").subscriber_id.isin(mail.subscriber_id).astype(int)
for c in ["emails_90d", "renewal_reminders_90d"]: F[c] = F[c].fillna(0)

# --- (vii) fan profile and merchandise
pr = prof.set_index("subscriber_id")
for c in ["age_band", "country", "marketing_consent", "has_favourite_team"]:
    F[c] = snap.set_index("row_id").subscriber_id.map(pr[c])
F["age_band"] = canon(F.age_band); F["country"] = canon(F.country)
mm = events_before(merch, "subscriber_id", "merch_purchase_datetime")
add(pd.DataFrame({"n_merch": mm.groupby("row_id").size(), "merch_spend": mm.groupby("row_id").merch_amount.sum()}))
F[["n_merch", "merch_spend"]] = F[["n_merch", "merch_spend"]].fillna(0)

# --- (viii) race-calendar context (the schedule is public, so upcoming races are legitimately known in advance)
rd = cal.event_date.values
o = snap.set_index("row_id").obs_date.values
idx = np.searchsorted(rd, o)
F["days_since_last_race"] = [(o[i] - rd[j - 1]) / np.timedelta64(1, "D") if j > 0 else np.nan for i, j in enumerate(idx)]
F["days_to_next_race"] = [(rd[j] - o[i]) / np.timedelta64(1, "D") if j < len(rd) else np.nan for i, j in enumerate(idx)]
F["races_next_60d"] = [int(((rd >= o[i]) & (rd < o[i] + np.timedelta64(60, "D"))).sum()) for i in range(len(o))]

data = snap[["row_id", "subscriber_id", "subscription_id", "obs_date", "y"]].set_index("row_id").join(F)
print("Feature table:", data.shape, "| numeric+categorical features:", data.shape[1] - 4)
data.head(3).T.head(30)

In [ ]:
# ---- 2d. Leakage guard ----------------------------------------------------------------------------------------------
FORBIDDEN = {"status", "end_datetime", "cancellation_datetime", "cancellation_type", "cancellation_reason", "auto_renew", "notification_setting"}
assert not FORBIDDEN & set(data.columns), "a leaking column slipped into the feature table"
ID_COLS = ["subscriber_id", "subscription_id", "obs_date", "y"]
CAT_COLS = ["plan_type", "acquisition_channel", "last_payment_method", "age_band", "country"]
NUM_COLS = [c for c in data.columns if c not in ID_COLS + CAT_COLS]
# plain numpy dtypes (float / object with np.nan) so scikit-learn and XGBoost never see pandas' pd.NA
data[NUM_COLS] = data[NUM_COLS].astype(float)
for c in CAT_COLS: data[c] = data[c].astype(object).where(data[c].notna(), np.nan)
print(f"{len(NUM_COLS)} numeric and {len(CAT_COLS)} categorical features")
print("Missing-value share (top):"); print(data[NUM_COLS + CAT_COLS].isna().mean().sort_values(ascending=False).head(8).round(3).to_string())

In [ ]:
# ---- 2e. Train / validation / test split: strictly by TIME (earlier months train, later months test) --------------------
VALID_START, TEST_START = pd.Timestamp("2025-01-01"), pd.Timestamp("2025-07-01")
emb = pd.Timedelta(days=HORIZON)    # embargo: a row's label window must finish before the next subset starts
train = data[(data.obs_date < VALID_START) & (data.obs_date + emb <= VALID_START)]
valid = data[(data.obs_date >= VALID_START) & (data.obs_date < TEST_START) & (data.obs_date + emb <= TEST_START)]
test  = data[data.obs_date >= TEST_START]
dropped = len(data) - len(train) - len(valid) - len(test)
split_tbl = pd.DataFrame({"rows": [len(train), len(valid), len(test)],
    "subscribers": [x.subscriber_id.nunique() for x in (train, valid, test)],
    "obs. dates": [f"{x.obs_date.min():%b %Y} - {x.obs_date.max():%b %Y}" for x in (train, valid, test)],
    "positives": [int(x.y.sum()) for x in (train, valid, test)], "positive rate": [x.y.mean() for x in (train, valid, test)]},
    index=["train", "validation", "test (unseen)"])
print(f"{dropped} rows dropped by the embargo (their label windows would overlap the next subset)")
overlap = len(set(test.subscriber_id) & set(train.subscriber_id))
print(f"{overlap} of {test.subscriber_id.nunique()} test subscribers also appear (in earlier months) in training: realistic for deployment, "
      "because the same fans are re-scored every month; they are never scored on the same month.")
split_tbl.style.format({"positive rate": "{:.2%}"})

In [ ]:
# ---- 2f. Model-specific encoders (fitted on the TRAINING subset only) ------------------------------------------------
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, OrdinalEncoder, StandardScaler

def signed_log1p(x): return np.sign(x) * np.log1p(np.abs(x))

# XGBoost: handles NaN natively -> no imputation or scaling; categoricals ordinal-encoded (unseen -> -1).
tree_prep = ColumnTransformer([
    ("num", "passthrough", NUM_COLS),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="constant", fill_value="missing")),
                      ("ord", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1))]), CAT_COLS)])
# Logistic regression baseline: median impute (+ missing flags), signed log1p, standardise; one-hot categoricals.
lr_prep = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median", add_indicator=True)), ("log", FunctionTransformer(signed_log1p)),
                      ("sc", StandardScaler())]), NUM_COLS),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="constant", fill_value="missing")),
                      ("oh", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=100, sparse_output=False))]), CAT_COLS)])

X_cols = NUM_COLS + CAT_COLS
Xtr, ytr = train[X_cols], train.y.to_numpy()
Xva, yva = valid[X_cols], valid.y.to_numpy()
Xte, yte = test[X_cols],  test.y.to_numpy()
tree_prep.fit(Xtr); lr_prep.fit(Xtr)
Ttr, Tva, Tte = (tree_prep.transform(x) for x in (Xtr, Xva, Xte))
Ltr, Lva, Lte = (lr_prep.transform(x) for x in (Xtr, Xva, Xte))
print("XGBoost matrix:", Ttr.shape, "| logistic-regression matrix:", Ltr.shape)

---
# 3. MODEL TRAINING
*(Component 3 of 5)*

### Hyper-parameter choices and why

**Final method (from SS2): XGBoost gradient-boosted trees**, trained on the 2024 observations and tuned on the Jan-Jun 2025 validation months. XGBoost fits trees one after another by a second-order (Newton) gradient step on the log-loss; this is the "optimiser".

| Hyper-parameter | Value | Justification |
|---|---|---|
| `learning_rate` | 0.05 | Small steps over many trees generalise better than a large rate; this is the SS2 value. |
| `n_estimators` | up to 2000 with **early stopping after 50 rounds** without validation log-loss improvement | The number of epochs (trees) is not guessed; it is set by the validation curve (see learning-curve plot), which stopped at 363 trees in our run. |
| `max_depth` | grid {2, 3, 4, 6}, **chosen by validation log loss (selected 2)** | SS2 searched {4, 6, 8} on a dataset with hundreds of thousands of users. Here there are only ~600 training positives, so shallower trees were added to the grid; the validation score preferred the simplest one, which limits over-fitting. |
| `min_child_weight` | 5 | Stops leaves being built from a handful of subscribers. |
| `subsample`, `colsample_bytree` | 0.8, 0.8 | Row and column sub-sampling de-correlates trees and reduces variance. |
| `reg_lambda` | 1.0 | Default L2 shrinkage of leaf weights. |
| `tree_method` | `hist` | Fast histogram splits; no accuracy cost at this size. |
| Class weighting | none | Keeps probabilities calibrated (the retention team needs risk *scores*); SS2 evidence (Ahmad et al., 2019). |
| Selection metric | validation **log loss** (PR-AUC reported alongside) | A proper scoring rule that rewards calibrated probabilities; PR-AUC alone is noisy with ~140 validation positives. |
| Decision threshold | F1-optimal on the **validation** months (0.183 in our run), never tuned on test | Needed only for the confusion-matrix metrics; the ranked score is the main output. |
| Seed | 42 | Reproducibility. |

**Baselines (SS2 Model 1 and the SS1 rule)** give the result context: L2 logistic regression with `C` chosen from {0.001, 0.01, 0.1, 1, 10} by validation log loss (strong regularisation won), and the simple rule "no viewing in the last 30 days".

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, log_loss, roc_auc_score, precision_recall_curve
from xgboost import XGBClassifier

def best_f1_threshold(y, p):
    pr, rc, th = precision_recall_curve(y, p)
    f1 = 2 * pr[:-1] * rc[:-1] / np.clip(pr[:-1] + rc[:-1], 1e-12, None)
    return float(th[np.argmax(f1)])

# ---- Final method: XGBoost (SS2 Model 2) with the SS2 hyper-parameter design -------------------------------------------
FIXED = dict(learning_rate=0.05, n_estimators=2000, min_child_weight=5, subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
             tree_method="hist", eval_metric="logloss", early_stopping_rounds=50, random_state=SEED, n_jobs=-1)
DEPTH_GRID = [2, 3, 4, 6]
grid_log, best = [], None
for d in DEPTH_GRID:
    clf = XGBClassifier(max_depth=d, **FIXED)
    clf.fit(Ttr, ytr, eval_set=[(Ttr, ytr), (Tva, yva)], verbose=False)     # early stopping watches the LAST eval set (validation)
    p = clf.predict_proba(Tva)[:, 1]
    row = {"max_depth": d, "trees_used": clf.best_iteration + 1, "valid_log_loss": log_loss(yva, p),
           "valid_PR_AUC": average_precision_score(yva, p), "valid_ROC_AUC": roc_auc_score(yva, p)}
    grid_log.append(row)
    if best is None or row["valid_log_loss"] < best[0]: best = (row["valid_log_loss"], d, clf)
xgb_model, best_depth = best[2], best[1]
p_va_xgb = xgb_model.predict_proba(Tva)[:, 1]
THRESH = best_f1_threshold(yva, p_va_xgb)     # operating point chosen on validation data only
print(f"Selected max_depth={best_depth} with {xgb_model.best_iteration + 1} trees; F1-optimal threshold (validation) = {THRESH:.3f}")
pd.DataFrame(grid_log).set_index("max_depth").round(4)

In [ ]:
# ---- Baselines for context: logistic regression (SS2 Model 1) and the simple rule from the SS1 problem statement ----------
lr_log, best_lr = [], None
for C in [0.001, 0.01, 0.1, 1, 10]:
    lr = LogisticRegression(C=C, penalty="l2", solver="lbfgs", max_iter=2000).fit(Ltr, ytr)
    p = lr.predict_proba(Lva)[:, 1]
    row = {"C": C, "valid_log_loss": log_loss(yva, p), "valid_PR_AUC": average_precision_score(yva, p)}
    lr_log.append(row)
    if best_lr is None or row["valid_log_loss"] < best_lr[0]: best_lr = (row["valid_log_loss"], C, lr)
lr_model, best_C = best_lr[2], best_lr[1]
THRESH_LR = best_f1_threshold(yva, lr_model.predict_proba(Lva)[:, 1])
print(f"Logistic regression: chosen C={best_C}")

# Rule baseline: "no viewing in the last 30 days" -> flagged as at risk (score = days since last session, higher = riskier)
def rule_score(df): return df.days_since_last_session.to_numpy(float)
RULE_THRESH = 30
pd.DataFrame(lr_log).set_index("C").round(4)

In [ ]:
# ---- Training diagnostics: does the boosted model overfit? (log loss per boosting round, train vs validation) ---------
hist_xgb = xgb_model.evals_result()
fig, ax = plt.subplots(figsize=(6.2, 3.6))
ax.plot(hist_xgb["validation_0"]["logloss"], label="training", color="#1f77b4")
ax.plot(hist_xgb["validation_1"]["logloss"], label="validation", color="#d62728")
ax.axvline(xgb_model.best_iteration, ls="--", color="grey", lw=1, label=f"early stop ({xgb_model.best_iteration + 1} trees)")
ax.set_xlabel("boosting round"); ax.set_ylabel("log loss"); ax.set_title("XGBoost learning curve"); ax.legend()
fig.tight_layout(); fig.savefig(OUT_DIR / "figures" / "01_learning_curve.png", dpi=150); plt.show()